# DiffAM experiments on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lazypilottt/DiffAM-Driven-Adversarial-Makeup-Transfer-and-Face-Recognition-Robustness/blob/main/notebooks/diffam_colab_experiments.ipynb)

Runs from the course project, executed on a Colab Tesla T4:

1. Fine-tune the DiffAM makeup-removal model on the MT dataset.
2. Fine-tune adversarial makeup transfer on CelebA-HQ for three makeup styles and measure black-box attack success.
3. Single-image makeup removal and adversarial makeup transfer.
4. Build an ArcFace embedding cache for a gender-split reference pool used by the similarity-guided attack.

Logs are trimmed to losses, checkpoints and metrics (progress bars and library warnings removed).

## 1. Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install -q onnxruntime-gpu insightface lpips kagglehub

Upstream DiffAM, pinned to the commit used here. Pretrained weights, `assets.zip` and CelebAMask-HQ are read from Google Drive; download links are in the [DiffAM README](https://github.com/HansSunY/DiffAM).

In [ ]:
!git clone https://github.com/HansSunY/DiffAM
!git -C DiffAM checkout -q 323aef8

Cloning into 'DiffAM'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 91 (delta 9), reused 72 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 125.39 KiB | 1011.00 KiB/s, done.
Resolving deltas: 100% (9/9), done.


In [ ]:
%cd /content/DiffAM

/content/DiffAM


In [ ]:
!pip install -r requirements.txt
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
!unzip -o /content/drive/MyDrive/assets.zip -d /content/DiffAM/
!unzip -o /content/drive/MyDrive/CelebAMask-HQ.zip -d /content/DiffAM/assets/datasets/

In [ ]:
!rm -rf /content/DiffAM/pretrained
!cp -r /content/drive/MyDrive/pretrained/ /content/DiffAM/

## 2. Reference pool for similarity matching

The [Gender Classification Dataset](https://www.kaggle.com/datasets/cashutosh/gender-classification-dataset)
(training split) serves as the identity gallery, split into `boys/` and `girls/`.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("cashutosh/gender-classification-dataset")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/cashutosh/gender-classification-dataset/versions/1


In [ ]:
!ls /root/.cache/kagglehub/datasets/cashutosh/gender-classification-dataset/versions/1/Training/
!cp -r /root/.cache/kagglehub/datasets/cashutosh/gender-classification-dataset/versions/1/Training/ /content/DiffAM/

female	male


In [ ]:
!mv Training reference_pool
!mv reference_pool/male reference_pool/boys
!mv reference_pool/female reference_pool/girls

Embedding cache over the whole pool (47,009 faces). This run used a Colab helper script; the same step is now `python -m makeup_attack index` in this repository.

In [ ]:
!python cache_reference_embeddings.py

find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
Caching boys: 100% 23766/23766 [05:48<00:00, 68.14it/s]
Caching girls: 100% 23243/23243 [05:37<00:00, 68.82it/s]


## 3. Makeup removal fine-tuning

Fine-tunes the pretrained makeup diffusion model with CLIP guidance (`face with makeup` -> `face without makeup`)
plus identity, L1 and LPIPS losses. 50 training / 10 test images, 4 iterations, t0 = 150.

In [ ]:
!python main.py --makeup_removal --config MT.yml --exp ./runs/test --do_train 1 --do_test 1 --n_train_img 50 --n_test_img 10 --n_iter 4 --t_0 150 --n_inv_step 20 --n_train_step 3 --n_test_step 10 --lr_clip_finetune 8e-6 --model_path pretrained/makeup.pt

INFO - main.py - 2025-04-26 22:31:37,866 - Using device: cuda
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
INFO - main.py - 2025-04-26 22:31:37,868 - Exp instance id = 59082
INFO - main.py - 2025-04-26 22:31:37,868 - Exp comment = 
INFO - main.py - 2025-04-26 22:31:37,868 - Config =
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
./runs/test_MR_MT_t150_ninv20_ngen3_id1_l12_lr8e-06
   face with makeup
-> face without makeup
Original diffusion Model loaded.
Setting optimizer with lr=8e-06
Loading losses
Loading ResNet ArcFace
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/alex.pth
Prepare identity latent
precomputed/MT_train_t150_nim200_ninv20_pairs.pth
precomputed/MT_test_t150_nim200_ninv20_pairs.pth
Start finetuning
Sampling type: DDIM with eta 0.0
Uniform skip type
CHANGE face with makeup TO face without makeup
CLI

## 4. Adversarial makeup transfer

CelebA-HQ model fine-tuned per makeup style with the DiffAM makeup loss and an ensemble attack
(surrogates: FaceNet, IR152, IRSE50; black-box target: MobileFace, `--target_model 2`).
Attack success rate (ASR) is reported at FAR 0.1 / 0.01 / 0.001 on 10 test images.

**Note:** these runs use `--n_iter 2`. DiffAM keeps `loss_adv` out of the objective for the first `MT_iter_without_adv` (default 3) iterations, so the attack term was logged but never optimised here and the ASR below is a no-attack baseline. Use `--n_iter 4` or `--MT_iter_without_adv 1` to include it.

### Style XMY-254

In [ ]:
!python main.py --makeup_transfer --config celeba.yml --exp ./runs/test --do_train 1 --do_test 1 --n_train_img 50 --n_test_img 10 --n_iter 2 --t_0 30 --n_inv_step 10 --n_train_step 3 --n_test_step 1 --lr_clip_finetune 8e-6 --model_path pretrained/celeba_hq.ckpt --target_img 1 --target_model 2 --ref_img 'XMY-254'

INFO - main.py - 2025-04-28 14:49:08,008 - Using device: cuda
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
INFO - main.py - 2025-04-28 14:49:08,010 - Exp instance id = 25716
INFO - main.py - 2025-04-28 14:49:08,010 - Exp comment = 
INFO - main.py - 2025-04-28 14:49:08,010 - Config =
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
./runs/test_MT_CelebA_HQ_XMY-254_t30_ninv10_ngen3_dis1_dir0.3_lr8e-06
Transfer makeup style XMY-254
Original diffusion Model loaded.
Setting optimizer with lr=8e-06
Loading losses
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/alex.pth
Prepare identity latent
precomputed/CelebA_HQ_train_t30_nim200_ninv10_pairs.pth
train pairs exists
precomputed/CelebA_HQ_test_t30_nim200_ninv10_pairs.pth
test pairs exists
Start finetuning
Sampling type: DDIM with eta 0.0
Uniform skip type
CLIP 0-0: loss_adv:

### Style XYH-045

In [ ]:
!python main.py --makeup_transfer --config celeba.yml --exp ./runs/test --do_train 1 --do_test 1 --n_train_img 50 --n_test_img 10 --n_iter 2 --t_0 30 --n_inv_step 10 --n_train_step 3 --n_test_step 1 --lr_clip_finetune 8e-6 --model_path pretrained/celeba_hq.ckpt --target_img 1 --target_model 2 --ref_img 'XYH-045'

INFO - main.py - 2025-04-28 14:52:57,460 - Using device: cuda
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
INFO - main.py - 2025-04-28 14:52:57,462 - Exp instance id = 26703
INFO - main.py - 2025-04-28 14:52:57,462 - Exp comment = 
INFO - main.py - 2025-04-28 14:52:57,462 - Config =
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
./runs/test_MT_CelebA_HQ_XYH-045_t30_ninv10_ngen3_dis1_dir0.3_lr8e-06
Transfer makeup style XYH-045
Original diffusion Model loaded.
Setting optimizer with lr=8e-06
Loading losses
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/alex.pth
Prepare identity latent
precomputed/CelebA_HQ_train_t30_nim200_ninv10_pairs.pth
train pairs exists
precomputed/CelebA_HQ_test_t30_nim200_ninv10_pairs.pth
test pairs exists
Start finetuning
Sampling type: DDIM with eta 0.0
Uniform skip type
CLIP 0-0: loss_adv:

### Style vRX912

In [ ]:
!python main.py --makeup_transfer --config celeba.yml --exp ./runs/test --do_train 1 --do_test 1 --n_train_img 50 --n_test_img 10 --n_iter 2 --t_0 30 --n_inv_step 10 --n_train_step 3 --n_test_step 1 --lr_clip_finetune 8e-6 --model_path pretrained/celeba_hq.ckpt --target_img 1 --target_model 2 --ref_img 'vRX912'

INFO - main.py - 2025-04-28 14:57:25,824 - Using device: cuda
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
INFO - main.py - 2025-04-28 14:57:25,825 - Exp instance id = 27878
INFO - main.py - 2025-04-28 14:57:25,826 - Exp comment = 
INFO - main.py - 2025-04-28 14:57:25,826 - Config =
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
./runs/test_MT_CelebA_HQ_vRX912_t30_ninv10_ngen3_dis1_dir0.3_lr8e-06
Transfer makeup style vRX912
Original diffusion Model loaded.
Setting optimizer with lr=8e-06
Loading losses
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/alex.pth
Prepare identity latent
precomputed/CelebA_HQ_train_t30_nim200_ninv10_pairs.pth
train pairs exists
precomputed/CelebA_HQ_test_t30_nim200_ninv10_pairs.pth
test pairs exists
Start finetuning
Sampling type: DDIM with eta 0.0
Uniform skip type
CLIP 0-0: loss_adv: 0

## 5. Single-image edits

Makeup removal (t0 = 300, 40 inversion steps).

In [ ]:
# makeup removal
!python main.py --edit_one_image_MR --config MT.yml --exp ../ --n_iter 1 --t_0 300 --n_inv_step 40 --n_train_step 6 --n_test_step 40 --img_path ../ram.jpg --model_path pretrained/makeup.pt

INFO - main.py - 2025-04-28 13:46:05,851 - Using device: cuda
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
INFO - main.py - 2025-04-28 13:46:05,853 - Exp instance id = 9573
INFO - main.py - 2025-04-28 13:46:05,853 - Exp comment = 
INFO - main.py - 2025-04-28 13:46:05,853 - Config =
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
None is loaded.
pretrained/makeup.pt is loaded.
Sampling type: DDIM with eta 0.0,  Steps: 40/300
Uniform skip type
total_time:17.185990571975708


Adversarial makeup transfer with the XMY-060 checkpoint (t0 = 60, 6 generative steps).

In [ ]:
!python main.py --edit_one_image_MT --config celeba.yml --exp ../ --n_iter 1 --t_0 60 --n_inv_step 20 --n_train_step 6 --n_test_step 6 --img_path ../ram.jpg --model_path checkpoint/test_MT_CelebA_HQ_XMY-060_t30_ninv10_ngen3_dis1_dir0.3_lr8e-06_XMY-060-1.pth

INFO - main.py - 2025-04-28 14:29:17,936 - Using device: cuda
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
INFO - main.py - 2025-04-28 14:29:17,940 - Exp instance id = 20572
INFO - main.py - 2025-04-28 14:29:17,940 - Exp comment = 
INFO - main.py - 2025-04-28 14:29:17,940 - Config =
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
Aligned image has shape: (256, 256)
None is loaded.
checkpoint/test_MT_CelebA_HQ_XMY-060_t30_ninv10_ngen3_dis1_dir0.3_lr8e-06_XMY-060-1.pth is loaded.
Sampling type: DDIM with eta 0.0,  Steps: 6/60
Uniform skip type
total_time:19.16168761253357


## 6. Export checkpoints

In [ ]:
!zip -r final_models.zip checkpoint/

updating: checkpoint/ (stored 0%)
  adding: checkpoint/.ipynb_checkpoints/ (stored 0%)
  adding: checkpoint/test_MT_CelebA_HQ_vFG137_t30_ninv10_ngen3_dis1_dir0.3_lr8e-06_vFG137-1.pth (deflated 8%)
  adding: checkpoint/test_MT_CelebA_HQ_vFG137_t30_ninv10_ngen3_dis1_dir0.3_lr8e-06_vFG137-0.pth (deflated 8%)


## 7. Similarity-guided attack

Pick the attack budget from the input face's nearest same-gender identity in the reference pool, then launch
DiffAM on the input. See the repository README for details.

In [ ]:
!pip install -q git+https://github.com/Lazypilottt/DiffAM-Driven-Adversarial-Makeup-Transfer-and-Face-Recognition-Robustness.git
!python -m makeup_attack index --gallery /content/DiffAM/reference_pool --out /content/cache/gallery.npz
!python -m makeup_attack run --image ../ram.jpg --index /content/cache/gallery.npz \
    --diffam-dir /content/DiffAM \
    --checkpoint checkpoint/test_MT_CelebA_HQ_XMY-060_t30_ninv10_ngen3_dis1_dir0.3_lr8e-06_XMY-060-1.pth